# Escenas L1C a procesar con ACOLITE (matchups con geometria)

Con los matchups nuevos (`matchups_registros_finales_geometria.geojson`) solo hay que correr ACOLITE en las escenas que no tienen un resultado utilizable:

1. **escenas nuevas**: su `pass_id` no estaba en los matchups anteriores
2. **escenas con puntos fuera del recorte**: ya se procesaron, pero ahora tienen puntos que antes no estaban y el L2W recortado no los cubre

El resto de las escenas siguen usando los resultados que ya existen. Para las escenas elegidas se busca el producto L1C en CDSE (misma logica que `09_acolite/descarga_l1c_scenes.ipynb`).

Salida: `data/nuevos/l1c_scene_products_geometria.csv` (name,id,online)

In [1]:
import sys
from pathlib import Path

_REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(_REPO_ROOT))
from core.cdse import find_l1c, sensing_and_tile


import geopandas as gpd
import numpy as np
import pandas as pd
import pystac
import requests
from tqdm.auto import tqdm

from core.ac_tools import dist_punto_pixeles, open_l2w
from tools.ac.extract_windows import ACOLITE_SAVE_DIR, MAX_DIST_PIXEL_M

/home/enzotrindade/ProyectoAA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MATCHUPS_NUEVOS = "../../data/nuevos/matchups_registros_finales_geometria.geojson"
MATCHUPS_ANTERIORES = "../../data/nuevos/matchups_registros_finales.geojson"
ITEMS_FILE = Path("../../data/nuevos/matchup_items_geometria.json")
OUT_FILE = "../../data/nuevos/l1c_scene_products_geometria.csv"
ACOLITE_DIR = Path("../..") / ACOLITE_SAVE_DIR

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

nuevos = gpd.read_file(MATCHUPS_NUEVOS)
anteriores = gpd.read_file(MATCHUPS_ANTERIORES)
items = {it.id: it for it in pystac.ItemCollection.from_file(str(ITEMS_FILE))}
print(len(nuevos), len(anteriores), len(items))

2957 2963 536


#### 1. Escenas nuevas

In [3]:
escenas_nuevas = sorted(set(nuevos["pass_id"]) - set(anteriores["pass_id"]))
print(len(escenas_nuevas), "escenas nuevas,", int(nuevos["pass_id"].isin(escenas_nuevas).sum()), "puntos")

84 escenas nuevas, 779 puntos


#### 2. Escenas ya procesadas con puntos fuera del recorte
Los puntos que no estaban en los matchups anteriores para esa escena se buscan en los L2W que ya estan en `data/acolite/`: si el pixel mas cercano esta a mas de `MAX_DIST_PIXEL_M`, el punto cae fuera del recorte.

In [4]:
# un punto se identifica por sus coordenadas (UTM, al decimetro)
def clave(d):
    return d.geometry.x.round(1).astype(str) + "_" + d.geometry.y.round(1).astype(str)

nuevos["clave"] = clave(nuevos)
anteriores["clave"] = clave(anteriores)
puntos_antes = anteriores.groupby("pass_id")["clave"].apply(set)

nuevos_wgs84 = nuevos.to_crs("EPSG:4326")
nuevos["lon"] = nuevos_wgs84.geometry.x
nuevos["lat"] = nuevos_wgs84.geometry.y

fuera_del_recorte = {}   # pass_id -> cantidad de puntos nuevos que el L2W actual no cubre
total_nuevos = 0
for pass_id, g in nuevos.groupby("pass_id"):
    if pass_id in escenas_nuevas:
        continue
    puntos_nuevos = g[~g["clave"].isin(puntos_antes[pass_id])]
    if puntos_nuevos.empty:
        continue
    total_nuevos += len(puntos_nuevos)

    # una escena dividida tiene un L2W por parte
    productos = [open_l2w(path) for path in sorted((ACOLITE_DIR / pass_id).glob("**/*L2W.nc"))]
    for _, p in puntos_nuevos.iterrows():
        dist_m = min((np.sqrt(dist_punto_pixeles(p, dl["lat"].values, dl["lon"].values).min()) * 111_320 for dl in productos), default=np.inf)
        if dist_m > MAX_DIST_PIXEL_M:
            fuera_del_recorte[pass_id] = fuera_del_recorte.get(pass_id, 0) + 1

print(total_nuevos, "puntos nuevos en escenas ya procesadas")
print(sum(fuera_del_recorte.values()), "puntos fuera del recorte, en", len(fuera_del_recorte), "escenas")

235 puntos nuevos en escenas ya procesadas
142 puntos fuera del recorte, en 26 escenas


In [5]:
escenas_a_correr = sorted(escenas_nuevas + list(fuera_del_recorte))
print(len(escenas_a_correr), "escenas a correr de", nuevos["pass_id"].nunique())

110 escenas a correr de 518


#### Productos L1C en CDSE

In [7]:
products = []
for pass_id in tqdm(escenas_a_correr, desc="Matching L1C"):
    item = items.get(pass_id)
    if item is None:
        print(f"[{pass_id}] not in {ITEMS_FILE.name}, skipping")
        continue
    stamp, tile = sensing_and_tile(item)
    try:
        l1c = find_l1c(stamp, tile)
    except requests.RequestException as e:
        print(f"[{pass_id}] CDSE query failed: {e}")
        continue
    if l1c is None:
        print(f"[{pass_id}] no L1C found for {stamp} T{tile}")
        continue
    products.append(l1c)

Matching L1C:   0%|          | 0/110 [00:00<?, ?it/s]

Matching L1C:   1%|          | 1/110 [00:12<22:52, 12.59s/it]

Matching L1C:   2%|▏         | 2/110 [00:24<22:25, 12.46s/it]

Matching L1C:   3%|▎         | 3/110 [00:31<17:29,  9.80s/it]

Matching L1C:   4%|▎         | 4/110 [00:45<20:25, 11.56s/it]

...

Matching L1C:  97%|█████████▋| 107/110 [15:10<00:19,  6.37s/it]

Matching L1C:  98%|█████████▊| 108/110 [15:17<00:13,  6.55s/it]

Matching L1C:  99%|█████████▉| 109/110 [15:22<00:06,  6.09s/it]

Matching L1C: 100%|██████████| 110/110 [15:33<00:00,  7.61s/it]

Matching L1C: 100%|██████████| 110/110 [15:33<00:00,  8.49s/it]

In [8]:
products_df = pd.DataFrame(products)
print(len(products_df), "de", len(escenas_a_correr), "| online:", int(products_df["online"].sum()))

# run_acolite.py empareja por (satelite, hora, tile), no por el nombre completo
clave_escena = lambda n: (n.split("_")[0], n.split("_")[2], n.split("_")[5])
encontradas = {clave_escena(n) for n in products_df["name"]}
faltan = [p for p in escenas_a_correr if clave_escena(p) not in encontradas]
print("sin producto L1C:", faltan)
products_df.head(2)

110 de 110 | online: 110
sin producto L1C: []


,name,id,online
0,S2A_MSIL1C_20170223T135111_N0500_R024_T21HUC_2...,788ddfbf-577a-46fe-8b64-a404d0def8f1,True
1,S2A_MSIL1C_20170504T140201_N0500_R024_T21HUC_2...,c17a4710-1821-4858-88fa-306f2220a35e,True


In [9]:
products_df.to_csv(OUT_FILE, index=False)